# 轻量语言模型全阶段训练与 Agentic RLVR：可执行工作报告

本 Notebook 按简历四条要求，把**真实源码、数学原理、实验结果和结论边界**连接起来。它只使用 Python 标准库读取仓库文件，因此在本地无 GPU 环境也能运行。

完整逐方法讲解见 [`07_RESUME_ALIGNED_TECHNICAL_REPORT.md`](../docs/foundation_model_interview/07_RESUME_ALIGNED_TECHNICAL_REPORT.md)。Notebook 负责从机器证据重新取数，避免把手抄表格当作实验事实。

> 阅读原则：实现正确、机制生效、固定集能力提升是三个不同层级。当前四算法完成前两层，但小预算固定集主要指标没有提升。

In [ ]:
from pathlib import Path
import csv
import json

repo = Path.cwd().resolve()
if not (repo / 'trainer').exists():
    repo = repo.parent
assert (repo / 'trainer' / 'policy_optimization.py').exists(), '请从仓库根目录或 learning_notebooks 目录运行'

evidence = repo / 'out' / 'remote_final' / 'evidence' / 'out'
assert evidence.exists(), f'最终证据目录不存在: {evidence}'

def read_json(relative_path):
    return json.loads((evidence / relative_path).read_text(encoding='utf-8'))

def read_csv(relative_path):
    with (evidence / relative_path).open(encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def show_between(relative_path, start_marker, end_marker=None, max_lines=220):
    text = (repo / relative_path).read_text(encoding='utf-8')
    start = text.index(start_marker)
    end = text.index(end_marker, start) if end_marker else len(text)
    lines = text[start:end].splitlines()[:max_lines]
    print('\n'.join(lines))

def print_rows(rows, columns):
    widths = {c: max(len(c), *(len(str(row.get(c, ''))) for row in rows)) for c in columns}
    print(' | '.join(c.ljust(widths[c]) for c in columns))
    print('-+-'.join('-' * widths[c] for c in columns))
    for row in rows:
        print(' | '.join(str(row.get(c, '')).ljust(widths[c]) for c in columns))

print('repo     =', repo)
print('evidence =', evidence)

## 0. 最终证据是否完整

证据索引保存 commit、环境、数据哈希、训练日志、评测 CSV、逐轨迹 JSONL 和最终验收状态。`missing=[]` 是本报告继续分析的前提。

In [ ]:
manifest = read_json('run_meta/experiment_evidence.json')
print('missing =', manifest.get('missing'))
assert manifest.get('missing') == []

base_commit = (evidence / 'run_meta' / 'base_commit.txt').read_text().strip()
torch_env = (evidence / 'run_meta' / 'torch_environment.txt').read_text().strip()
print('base commit:', base_commit)
print(torch_env)

# 一、轻量 Decoder-Only 架构

一个 Pre-Norm Block 是：`RMSNorm → Causal Attention → Residual → RMSNorm → SwiGLU/MoE → Residual`。

- **RMSNorm**：只按均方根缩放，归一化部分用 FP32。
- **RoPE/YaRN**：旋转 Q/K 注入相对位置；YaRN平滑调整长位置频率。
- **GQA**：8 个 Query heads 共享 4 个 KV heads，理论 KV Cache 减半。
- **KV Cache**：缓存历史 K/V，增量解码只计算新 token。
- **Flash**：通过 PyTorch SDPA 调度；必须用 profiler 证明实际命中。
- **SwiGLU**：`down(SiLU(gate(x)) * up(x))`。
- **MoE**：4 experts、Top-1 routing，并加入负载均衡辅助损失。

In [ ]:
# 直接展示真实 RMSNorm 与 RoPE/YaRN 源码；源码内已有逐行中文注释。
show_between('model/model_minimind.py', 'class RMSNorm', 'class Attention', max_lines=150)

In [ ]:
# Attention 中重点看：GQA 的 K/V projection、past_key_value 保存位置、
# repeat_kv、cached single-token SDPA fast path 和 causal mask。
show_between('model/model_minimind.py', 'class Attention', 'class FeedForward', max_lines=120)

In [ ]:
# SwiGLU 与 MoE：Router 只把 token 发送给 Top-1 Expert；
# aux_loss 约束专家负载，避免所有 token 长期路由到少数专家。
show_between('model/model_minimind.py', 'class FeedForward', 'class MiniMindBlock', max_lines=180)

### 架构真实结果

下面从最终 CSV/JSON 读取结果。结果边界：4096-token 基准证明 RoPE/YaRN 路径可运行，不证明长文本任务准确率提升；MoE 只部分训练，不证明质量或效率优于 Dense。

In [ ]:
arch = read_csv('architecture_benchmark_long_clean.csv')
print_rows(arch, [
    'weight', 'use_moe', 'total_parameters',
    'active_parameters_per_token_approx', 'gqa_cache_reduction_ratio',
    'cached_tokens_per_second', 'cached_peak_memory_mib'
])

print('\n4096-token RoPE/YaRN:')
yarn = read_csv('architecture_yarn_4096.csv')
print_rows(yarn, ['yarn', 'cached_tokens_per_second', 'uncached_tokens_per_second', 'cached_peak_memory_mib'])

flash = read_json('flash_sdpa_verification.json')
print('\nFlash speedup:', round(flash['flash_speedup_over_math'], 4), 'x')
print('Flash/math peak MiB:', flash['forced_flash']['peak_memory_mib'], '/', flash['forced_math']['peak_memory_mib'])
assert flash['profile_contains_flash_kernel']

moe = read_json('run_meta/moe_routing_audit.json')
print('MoE aggregate load:', [round(x, 4) for x in moe['aggregate_load_ratio']])
print('MoE load CV:', round(moe['aggregate_load_cv'], 4))
assert moe['router_gradients']['all_finite'] and moe['expert_gradients']['all_finite']

# 二、全阶段训练链路

实际依赖是 `Pretrain → Full SFT`，随后从 SFT checkpoint 分出 LoRA、KD 和 Agent SFT 支线。

- Pretrain：所有非 padding token 参与 next-token CE。
- SFT：只有 assistant token label 不为 `-100`。
- AMP：主干 BF16，敏感的 log-softmax/KL 转 FP32。
- 梯度累积：micro loss 除以 K，最后不完整窗口需要 remainder 修正。
- LoRA：冻结基模，只训练低秩 `BA`。
- KD：`α CE + (1-α) T² KL(teacher || student)`。

> 重要差距：当前真实学习率函数是 cosine decay，没有 Warmup。没有补代码和新日志前，简历应写“余弦学习率衰减”。

In [ ]:
# Assistant-only mask：默认 labels=-100，只打开 assistant span。
show_between('dataset/lm_dataset.py', 'class SFTDataset', 'class DPODataset', max_lines=125)

print('\n--- 当前学习率函数：没有 warmup ---')
show_between('trainer/trainer_utils.py', 'def get_lr', 'def init_distributed_mode', max_lines=30)

In [ ]:
# LoRA 中 B 零初始化保证初始 ΔW=0；merge 时执行 W += B @ A。
show_between('model/model_lora.py', 'class LoRA', 'def load_lora', max_lines=65)

print('\n--- KD：FP32 log-target KL 与 assistant mask ---')
show_between('trainer/train_distillation.py', 'def distillation_loss', 'if __name__ ==', max_lines=125)

In [ ]:
lora = read_json('eval/lora_medical_holdout_metrics_fixed.json')
kd = read_json('eval/kd_comparison.json')
mask = read_json('run_meta/agent_sft_mask_audit_128.json')

print('LoRA trainable params: 393,216 / 0.61%')
print('LoRA holdout loss delta %:', round(lora['delta']['loss_relative_percent'], 4))
print('LoRA holdout PPL delta % :', round(lora['delta']['perplexity_relative_percent'], 4))
print('KD parameter reduction % :', round(kd['compression']['parameter_reduction_percent'], 4))
print('KD vs CE PPL delta %     :', round(kd['kd_vs_ce']['perplexity_relative_percent'], 4))
print('Agent SFT mask mismatch/zero:', mask['mismatch_count'], '/', mask['zero_supervision_count'])

assert lora['delta']['perplexity_relative_percent'] < 0
assert kd['kd_vs_ce']['perplexity_relative_percent'] > 0  # 当前 KD 是负结果
assert mask['mismatch_count'] == mask['zero_supervision_count'] == 0

# 三、DPO：离线偏好优化分支

DPO 使用共享 prompt 的 `chosen/rejected` 回答，直接优化策略相对冻结 reference 的隐式奖励差：

`L = -log sigmoid(β[(logπ(chosen)-logπ(rejected)) - (logπref(chosen)-logπref(rejected))])`

它不需要在线 rollout、Reward Model 或 Critic，但仍需 policy/reference 两次前向。DPO 是 SFT 后的独立离线分支；本项目 GRPO 家族从 `agent_sft` 初始化，并不是从 DPO checkpoint 继续训练。

In [ ]:
# DPO 数据：chosen/rejected 共享 prompt，只累计 assistant span 的 log-prob。
show_between('dataset/lm_dataset.py', 'class DPODataset', 'class RLAIFDataset', max_lines=105)

print('\n--- DPO objective：policy margin 减 reference margin ---')
show_between('trainer/train_dpo.py', 'def logits_to_log_probs', 'def train_epoch', max_lines=45)

print('\n--- Prompt-grouped split：防止同 prompt 的 pairs 跨 train/eval ---')
show_between('scripts/prepare_dpo_data.py', 'def canonical_prompt', 'def main', max_lines=35)

In [ ]:
dpo_split = read_json('run_meta/dpo_split_manifest.json')
dpo_eval = read_json('eval/dpo_holdout_metrics.json')
dpo_rows = [
    {
        'model': dpo_eval['base']['checkpoint'],
        'pairs': dpo_eval['base']['pairs'],
        'preference_acc': round(dpo_eval['base']['chosen_preference_accuracy'], 6),
        'implicit_margin': round(dpo_eval['base']['mean_implicit_reward_margin'], 6),
        'positive_margin': round(dpo_eval['base']['positive_implicit_reward_margin_rate'], 6),
        'dpo_loss': round(dpo_eval['base']['dpo_loss_against_reference'], 6),
    },
    {
        'model': dpo_eval['dpo']['checkpoint'],
        'pairs': dpo_eval['dpo']['pairs'],
        'preference_acc': round(dpo_eval['dpo']['chosen_preference_accuracy'], 6),
        'implicit_margin': round(dpo_eval['dpo']['mean_implicit_reward_margin'], 6),
        'positive_margin': round(dpo_eval['dpo']['positive_implicit_reward_margin_rate'], 6),
        'dpo_loss': round(dpo_eval['dpo']['dpo_loss_against_reference'], 6),
    },
]
print('split train/eval/overlap:', dpo_split['train_samples'], dpo_split['eval_samples'], dpo_split['prompt_overlap'])
print_rows(dpo_rows, ['model', 'pairs', 'preference_acc', 'implicit_margin', 'positive_margin', 'dpo_loss'])
print('holdout accuracy delta points:', dpo_eval['changes']['chosen_preference_accuracy_points'])
print('holdout DPO loss delta %:', round(dpo_eval['changes']['dpo_loss_percent'], 6))

assert dpo_split['prompt_overlap'] == 0
assert dpo_eval['changes']['chosen_preference_accuracy_points'] == 0.0
assert dpo_eval['changes']['dpo_loss_percent'] > 0  # 当前配置是负结果

### DPO 暴露的问题与后续处理

- 离线 pairs 无法覆盖当前策略新产生的错误；后续 GRPO 家族改为在线采样。
- 单个 pair label 不能区分工具格式、参数、执行、证据和最终答案；后续 Agent verifier 分项验证。
- 序列 log-prob 直接求和对 chosen/rejected 长度敏感；本项目尚未完成 length-normalized DPO 消融。
- DPO 不需要 Critic，但仍需冻结 reference；它降低系统复杂度，不是零成本。
- 本次 1,000 对 holdout 的准确率仍为 45.70%，DPO loss 上升 0.224%，因此不能宣传偏好提升。

后续方法解决了反馈和优化机制问题：Agent SFT 负责协议冷启动，在线 RLVR 负责执行当前轨迹，GRPO 去掉 Critic，DAPO 过滤零方差组，GSPO 对齐序列级 Reward；但当前小预算固定集仍未涨点。

# 四、GRPO / CISPO / DAPO / GSPO

共同组相对优势：`A=(r-group_mean)/(group_std+eps)`。全对/全错组的优势全部为 0。

| 算法 | Importance Ratio 粒度 | Clip | Reduction | 关键特点 |
|---|---|---|---|---|
| GRPO | Token | 对称 | sequence-first | 无 Critic 基线 |
| CISPO | Token coefficient | 单侧上界 | global token mean | coefficient stop-gradient，直接优化 log-prob |
| DAPO | Token | 非对称 | global token mean | dynamic sampling + clip-higher + soft overlong |
| GSPO | Sequence | 序列级 | sequence mean | token ratios 的长度归一化几何平均 |

DAPO 的“稳定性改善”必须限定为：减少被用于更新的零方差组、恢复有效相对优势；它不等于固定集性能或总体跨 seed 稳定性已经提高。

In [ ]:
# 动态采样、Soft Overlong、正 KL 和四算法统一目标。
# 阅读时重点跟踪 [N,T] 的 current/old/reference logps 与 completion_mask。
show_between('trainer/policy_optimization.py', 'def group_relative_advantages', None, max_lines=205)

### 训练组结果与固定集结果不能混用

DAPO 只保留混合成功组，因此训练组 Reward/Accuracy 存在选择偏差。可信泛化比较必须看共同 Agent SFT baseline 和 12 个 RL checkpoints 的固定 holdout。

In [ ]:
for task in ['math', 'tool']:
    rows = read_csv(f'metrics/algorithm_comparison_{task}.csv')
    compact = []
    for row in rows:
        compact.append({
            'algorithm': row['algorithm'],
            'train_reward': round(float(row['reward_mean']), 4),
            'train_acc': round(float(row['task_accuracy_mean']), 4),
            'zero_var': round(float(row['zero_variance_group_rate_mean']), 4),
            'candidate_groups': round(float(row['candidate_groups_mean']), 1),
            'wall_s': round(float(row['wall_time_seconds_mean']), 2),
        })
    print(f'\n{task.upper()} training metrics')
    print_rows(compact, ['algorithm', 'train_reward', 'train_acc', 'zero_var', 'candidate_groups', 'wall_s'])

for task in ['math', 'tool']:
    rows = read_csv(f'eval_rlvr/{task}/algorithm_comparison.csv')
    compact = []
    for row in rows:
        compact.append({
            'algorithm': row['algorithm'],
            'reward': round(float(row['reward_mean']), 6),
            'accuracy': round(float(row['task_accuracy_mean']), 6),
            'kl': f"{float(row['kl_k3_mean']):.3e}",
            'avg_len': round(float(row['response_length_mean']), 2),
            'unfinished': round(float(row['unfinished_mean']), 6),
            'delta_acc': round(float(row['task_accuracy_delta_vs_agent_sft']), 6),
        })
    print(f'\n{task.upper()} fixed-holdout metrics')
    print_rows(compact, ['algorithm', 'reward', 'accuracy', 'kl', 'avg_len', 'unfinished', 'delta_acc'])

# 五、Agentic RL / RLVR

轨迹状态机是 `Assistant → Tool → Observation → Assistant`。Policy loss 只能覆盖 assistant action；observation 可以进入后续上下文，但 action mask 必须为 0。

严格成功同时要求：最终答案正确、标签闭合、工具名称与参数合法、调用执行成功、必需工具被覆盖、执行结果能支持 GT、轨迹已正常结束。Shaped reward 不能替代这个二值成功条件。

In [ ]:
# Verifier 核心：分别计算格式、调用合法性、执行、证据和最终答案；
# 最后才合成严格 task_success，防止格式奖励和猜答案造成 Reward Hacking。
show_between('trainer/train_agent.py', 'def calculate_rewards', '# ================================ 工具与 Reward = End', max_lines=175)

### 截断率的口径

当前正式表记录 `unfinished`、平均长度和 P95 长度，并禁止 rollout 后左截断。`unfinished` 表示达到最大轮数仍未完成，不等于命中 token 上限的 `truncated_rate`。在增加 `stop_reason={eos,max_new_tokens,max_turns,max_context,tool_error}` 前，简历应写“未完成率与 P95 响应长度”，不能把两者混写。

In [ ]:
paths = {
    'math_before': 'eval_rlvr/readiness_pre_sft_math/summary.csv',
    'math_after': 'eval_rlvr/readiness_math/summary.csv',
    'tool_before': 'eval_rlvr/readiness_pre_sft_tool/summary.csv',
    'tool_after': 'eval_rlvr/readiness_tool/summary.csv',
}
readiness = []
for name, path in paths.items():
    row = read_csv(path)[0]
    readiness.append({
        'stage': name,
        'task_acc': round(float(row['task_accuracy']), 6),
        'format': round(float(row['format_valid_rate']), 6),
        'tool_valid': round(float(row['tool_call_valid_rate']), 6),
        'execution': round(float(row['tool_execution_success_rate']), 6),
        'evidence': round(float(row['tool_evidence_coverage_rate']), 6),
        'unfinished': round(float(row['unfinished']), 6),
    })
print_rows(readiness, ['stage', 'task_acc', 'format', 'tool_valid', 'execution', 'evidence', 'unfinished'])

assert readiness[0]['task_acc'] == 0.0
assert readiness[1]['task_acc'] == 0.59375
assert readiness[2]['task_acc'] == 0.070312
assert readiness[3]['task_acc'] == 1.0

# 六、最终面试口径

1. **架构正结果**：GQA 理论 KV Cache -50%；指定 shape Flash SDPA 6.55×；4096-token 路径可运行。MoE 只证明容量/路由/成本。
2. **训练与偏好分支**：LoRA 0.61% 参数使医疗 holdout PPL -12.15%；Assistant-only mask 零错位；KD/DPO 是公平负对照，DPO preference accuracy 没有提升。当前没有 Warmup。
3. **RL 机制结果**：DAPO 让 Tool 保留组零方差率从普通采样的 86.7%–93.3% 降为 0%，但付出约 10.4×候选组和 9.9×时间。
4. **初始 Formal 结论**：Math/Tool 各 2,496 条轨迹中，四算法主要指标相对 Agent SFT delta=0；该轮不能宣传 RL 准确率提升。
5. **后续优化结论**：修复 FP16 checkpoint 舍入与行为 log-prob 对齐，并使用无 holdout 泄漏的 96 条类别平衡 Tool cold-start 后，严格成功率由 4.86% 提至 35.76%；同一课程 SFT 与 30-update 目标下，GRPO/CISPO/DAPO/GSPO 分别为 35.76%/35.07%/36.11%/35.76%。主要收益来自课程 SFT，DAPO 仅 +0.35pp 且 rollout 成本约 9.33 倍，不能声称统计显著或算法优越性。
6. **Agent 正结果**：Agent SFT 将 Math 从 0% 提至 59.375%、Basic Tool 从 7.031% 提至 100%；verifier 能拒绝数字子串、格式作弊、错误工具、伪造证据与未闭合轨迹。

完整推荐简历文字、DPO 问题链、公式推导、失败分析与高频追问见总报告；RL 不增长的根因、代码修复和后续优化数字见 `docs/foundation_model_interview/09_RL_OPTIMIZATION_FOLLOWUP.md`。